# Pipeline conversacional de Chatbot com NLP (chatbotv7)

- **Repositório:** `nlp-llms-portfolio`  
- **Tecnologias:** NLTK, spaCy, Scikit-Learn  
- **Arquivo Original:** `Cópia de ChatBotV7.ipynb`  

---


In [1]:
!pip install --upgrade pip
!pip install spacy
!python -m spacy download pt_core_news_lg
!pip install --upgrade numpy scipy gensim
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 24.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 143.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 46.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully

In [50]:
import spacy
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Carregar o modelo do spaCy
nlp = spacy.load("pt_core_news_lg")

In [51]:
def cosine_similarity_manual(vector1, vector2):
    """
    Calcula a similaridade de cosseno entre dois vetores NumPy densos.
    Assume que as entradas já foram convertidas para arrays NumPy.

    Args:
        vector1 (np.ndarray): O primeiro vetor. Pode ser 1D ou 2D.
        vector2 (np.ndarray): O segundo vetor. Pode ser 1D ou 2D.

    Returns:
        float: A similaridade de cosseno entre os dois vetores.
               Retorna 0.0 se um dos vetores for nulo (todo zero).
    """

    # Garante que os vetores são arrays NumPy e os achata para 1D
    # np.array() aqui é mais para segurança se o input não for np.ndarray
    v1 = np.array(vector1).flatten()
    v2 = np.array(vector2).flatten()

    # Calcula o produto escalar (dot product)
    dot_product = np.dot(v1, v2)

    # Calcula as normas (magnitudes) dos vetores
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    # Evita divisão por zero se um dos vetores for nulo
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0

    # Calcula a similaridade de cosseno
    similarity = dot_product / (norm_v1 * norm_v2)

    return similarity

In [52]:
import requests

url = 'https://raw.githubusercontent.com/VictorSilvaCamargo/o-guarani/main/o-guarani.txt'
response = requests.get(url)

if response.status_code == 200:
    texto_livro = response.text
else:
    raise Exception(f"Erro ao carregar o livro: {response.status_code}")


In [53]:
# --- Pre-processamento ---
def preprocessar_texto(texto):
    texto = texto.lower()
    doc = nlp(texto)
    tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]
    return " ".join(tokens)

# --- Geração de Chunks e Extração de Entidades ---
def processar_texto_e_gerar_chunks(texto, tamanho_chunk=200, sobreposicao=50):
    chunks = []
    mapa_chunks = {}
    doc = nlp(texto)
    sentences = [sent.text for sent in doc.sents] # Dividir em sentenças

    current_chunk_tokens = []
    current_chunk_text = []
    current_chunk_entities = {'PER': set(), 'LOC': set(), 'ORG': set(), 'MISC': set()}
    chunk_id_counter = 0

    for i, sent_text in enumerate(sentences):
        sent_doc = nlp(sent_text) # Processar sentença individualmente para entidades e tokens
        sent_tokens = [token.lemma_ for token in sent_doc if token.is_alpha and not token.is_stop]

        # Extrair entidades
        for ent in sent_doc.ents:
            if ent.label_ in current_chunk_entities:
                current_chunk_entities[ent.label_].add(ent.text)

        # Adicionar à lista de tokens e texto do chunk
        current_chunk_tokens.extend(sent_tokens)
        current_chunk_text.append(sent_text)

        # Verificar se o chunk atingiu o tamanho
        if len(current_chunk_tokens) >= tamanho_chunk:
            chunk_text_str = " ".join(current_chunk_text)
            chunk_tokens_str = " ".join(current_chunk_tokens) # Tokens para Word2Vec

            mapa_chunks[chunk_id_counter] = {
                'texto_original': chunk_text_str,
                'tokens_processados': chunk_tokens_str.split(), # Lista de tokens para Word2Vec
                'entidades': {k: list(v) for k, v in current_chunk_entities.items()} # Converte set para list
            }
            chunks.append(chunk_text_str) # Para TF-IDF

            chunk_id_counter += 1

            # Reiniciar para o próximo chunk com sobreposição
            # Mantemos parte do final do chunk anterior como início do próximo
            num_overlap_tokens = len(sent_tokens) # Mantenha a última sentença como sobreposição
            current_chunk_tokens = sent_tokens # Comece o novo chunk com os tokens da última sentença
            current_chunk_text = [sent_text] # Comece o novo chunk com a última sentença
            current_chunk_entities = {'PER': set(), 'LOC': set(), 'ORG': set(), 'MISC': set()}
            for ent in sent_doc.ents: # Re-adiciona entidades da última sentença
                if ent.label_ in current_chunk_entities:
                    current_chunk_entities[ent.label_].add(ent.text)

    # Adicionar o último chunk se houver
    if current_chunk_tokens:
        chunk_text_str = " ".join(current_chunk_text)
        chunk_tokens_str = " ".join(current_chunk_tokens)
        mapa_chunks[chunk_id_counter] = {
            'texto_original': chunk_text_str,
            'tokens_processados': chunk_tokens_str.split(),
            'entidades': {k: list(v) for k, v in current_chunk_entities.items()}
        }
        chunks.append(chunk_text_str)

    return chunks, mapa_chunks

In [54]:
# Dicionários de Entidades Curadas
DIC_PERSONAGENS = {
    "D. Antônio de Mariz": {"alias": ["Dom Antônio", "Antônio de Mariz", "D. Antônio"], "tipo": "Patriarca"},
    "D. Lauriana": {"alias": ["Dom Lauriana", "Lauriana"], "tipo": "Esposa"},
    "D. Diogo de Mariz": {"alias": ["Dom Diogo", "Diogo de Mariz", "Diogo"], "tipo": "Filho"},
    "D. Cecília": {"alias": ["Dom Cecília", "Cecília", "Ceci"], "tipo": "Filha"},
    "D. Isabel": {"alias": ["Dom Isabel", "Isabel"], "tipo": "Filha Natural"},
    "Aires Gomes": {"alias": ["Aires", "Gomes"], "tipo": "Escudeiro"},
    "Álvaro de Sá": {"alias": ["Álvaro", "Álvaro de Sa"], "tipo": "Cavalheiro"},
    "Loredano": {"alias": ["Frei Ângelo di Luca", "Frei Ângelo", "Ângelo di Luca"], "tipo": "Antagonista"},
    "Rui Soeiro": {"alias": ["Rui", "Soeiro"], "tipo": "Cúmplice"},
    "Bento Simões": {"alias": ["Bento", "Simões"], "tipo": "Cúmplice"},
    "Martim Vaz": {"alias": ["Martim", "Vaz"], "tipo": "Aventureiro"},
    "Vasco Afonso": {"alias": ["Vasco", "Afonso"], "tipo": "Aventureiro"},
    "João Feio": {"alias": ["João", "Feio"], "tipo": "Aventureiro"},
    "Mestre Nunes": {"alias": ["Nunes"], "tipo": "Colono"},
    "Peri": {"alias": ["Guerreiro Goitacá"], "tipo": "Herói Indígena"},
    "Ararê": {"alias": [], "tipo": "Pai de Peri"},
    "Cacique Aimoré": {"alias": ["Chefe Aimoré"], "tipo": "Antagonista Indígena"},
    "Mulher Aimoré": {"alias": ["Mulher Goitacá"], "tipo": "Membro da Tribo"}, # Cuidado com ambiguidades, pode ser necessário mais granularidade se for diferenciar as 2
    "Mãe Aimoré": {"alias": [], "tipo": "Mãe de Peri"},
    "Mem de Sá": {"alias": [], "tipo": "Governador"},
    "Dr. Antônio de Salema": {"alias": ["Antônio de Salema"], "tipo": "Líder de Expedição"},
    "D. Pedro da Cunha": {"alias": ["Pedro da Cunha"], "tipo": "Figura Política"},
    "D. Antônio, prior do Crato": {"alias": ["Prior do Crato"], "tipo": "Herdeiro Legítimo"},
    "D. Filipe II": {"alias": ["Filipe II"], "tipo": "Rei"},
    "Robério Dias": {"alias": [], "tipo": "Descobridor de Minas"},
    "Crispim Tenreiro": {"alias": [], "tipo": "Cunhado de D. Antônio"},
    "Martim de Sá": {"alias": [], "tipo": "Governador da Capitania"},
    "Garcia Ferreira": {"alias": [], "tipo": "Tabelião"},
}

DIC_LOCAIS = {
    "Serra dos Órgãos": {"alias": []},
    "Rio Paquequer": {"alias": ["Paquequer"]},
    "Rio Paraíba": {"alias": ["Paraíba"]},
    "Florestas": {"alias": ["Mata", "Sertão", "Selva", "Floresta"]}, # Adicione variações comuns
    "Casa de D. Antônio de Mariz": {"alias": ["Casa", "Fortaleza de Mariz", "Habitação de D. Antônio"]},
    "Cabana de Peri": {"alias": ["Cabana"]},
    "Rio de Janeiro": {"alias": ["Rio", "RJ", "Cidade do Rio de Janeiro"]},
    "Cabo Frio": {"alias": []},
    "Minas Gerais": {"alias": ["Minas", "Espírito Santo"]}, # Juntei, adapte se precisar separar estados
    "São Sebastião": {"alias": []},
    "Salvador": {"alias": []},
    "Madri": {"alias": ["Madrid"]},
    "Pouso": {"alias": []},
    "Taba dos brancos": {"alias": ["Assentamento branco", "Aldeia branca"]},
    "Campo dos Aimorés": {"alias": ["Acampamento Aimoré", "Taba dos Aimorés"]},
    "Despensa": {"alias": []},
    "Oratório": {"alias": []},
    "Gabinete d’armas": {"alias": ["Gabinete de armas", "Arsenal"]},
}

DIC_EVENTOS_PALAVRAS_CHAVE = {
    "Mulher Aimoré morta por Diogo": ["morte", "diogo", "mulher", "aimoré", "caçada"],
    "Plano de envenenamento de Peri": ["envenenamento", "peri", "aimoré", "plano"],
    "Explosão da casa de D. Antônio de Mariz": ["explosão", "casa", "d. antônio", "morte", "fim"],
    "Fuga de Peri e Cecília da inundação": ["fuga", "peri", "cecília", "inundação", "enchente"],
    "Batismo cristão de Peri": ["batismo", "cristão", "peri", "d. antônio"],
    "Decisão de Cecília de ficar com Peri na natureza": ["decisão", "cecília", "peri", "natureza", "viver"],
    "Confronto verbal entre Álvaro e Loredano": ["confronto", "álvaro", "loredano", "discussão", "tensão"],
    "Peri e a Onça": ["peri", "onça", "confronto", "habilidade", "floresta"],
    "Oração da Ave-Maria": ["oração", "ave-maria", "família", "religião", "crepúsculo"],
    "Tentativa de presente de Álvaro e recusa de Cecília": ["presente", "álvaro", "cecília", "recusa", "loredano"],
    "Confronto de Loredano com Rui Soeiro e Bento Simões": ["confronto", "loredano", "rui soeiro", "bento simões", "plano", "conspiração"],
    "Cecília confronta Peri sobre a onça": ["cecília", "peri", "onça", "confronto", "devoção"],
    "Cecília dá pistolas a Peri": ["cecília", "pistolas", "peri", "presente", "confiança"],
    "Isabel confessa amor por Álvaro": ["isabel", "álvaro", "amor", "confissão", "triângulo amoroso"],
    "Álvaro confessa amor por Isabel": ["álvaro", "isabel", "amor", "confissão", "dilema"],
    "Peri recupera o presente de Álvaro para Cecília": ["peri", "presente", "álvaro", "cecília", "recuperação", "dedicação"],
    "Dois guerreiros Aimorés mortos por Peri": ["morte", "aimorés", "peri", "proteção", "ataque"],
    "Cão morto por mulher Aimoré": ["morte", "cão", "mulher aimoré", "sacrifício", "fuga"],
    "Testamento de D. Antônio e revelação da verdadeira paternidade de Isabel": ["testamento", "d. antônio", "isabel", "paternidade", "revelação"],
    "Plano de D. Antônio para explodir a casa": ["plano", "d. antônio", "explosão", "casa", "pólvora"],
    "Partida de D. Diogo para São Sebastião": ["partida", "d. diogo", "são sebastião", "reforços", "missão"],
    "Confronto de D. Antônio com os aventureiros rebeldes": ["confronto", "d. antônio", "aventureiros", "revolta", "autoridade"],
    "Álvaro expulsa Loredano": ["expulsão", "álvaro", "loredano", "ameaça", "conspiração"],
    "Mestre Nunes revela a identidade de Loredano a Aires Gomes": ["revelação", "loredano", "mestre nunes", "aires gomes", "identidade"],
    "Peri mata Bento Simões e Rui Soeiro": ["morte", "peri", "bento simões", "rui soeiro", "assassinato", "proteção"],
    "Loredano é amarrado e queimado": ["morte", "loredano", "queimado", "punição", "traição"],
    "Ataque dos Aimorés à casa": ["ataque", "aimorés", "casa", "incêndio", "vingança"],
    "Batalha de Álvaro e seus homens contra os Aimorés": ["batalha", "álvaro", "aimorés", "confronto", "defesa"],
    "Peri ataca o acampamento Aimoré e envenena a tribo": ["ataque", "peri", "aimoré", "envenenamento", "extermínio"],
    "Cacique Aimoré morto por Álvaro": ["morte", "cacique aimoré", "álvaro", "batalha", "líder"],
    "Mulher Aimoré (esposa de Peri) morta por flecha": ["morte", "mulher aimoré", "peri", "sacrifício", "flecha"],
    "Morte implícita de Álvaro": ["morte", "álvaro", "batalha", "tragédia"],
    "Fuga de Peri e Cecília da casa em chamas": ["fuga", "peri", "cecília", "incêndio", "salvação"],
    "Loredano busca hospitalidade na casa de D. Antônio": ["loredano", "hospitalidade", "d. antônio", "infiltração"],
    "Isabel possui e contempla o veneno curare": ["isabel", "curare", "veneno", "desespero", "segredo"],
    "Peri é capturado pelos Aimorés": ["peri", "captura", "aimorés", "sacrifício", "ritual"],
    "Peri inunda os aposentos dos aventureiros": ["peri", "inundação", "aposentos", "aventureiros", "estratégia"],
    "Cecília compreende o significado do nome 'Ceci'": ["cecília", "ceci", "significado", "remorso", "compreensão"],
    "O ritual de sacrifício Aimoré e a oferta da arma a Peri": ["ritual", "sacrifício", "aimoré", "peri", "arma", "libertação"],
    "D. Antônio reprime a revolta dos aventureiros": ["d. antônio", "repressão", "revolta", "aventureiros", "autoridade"]
}

# Para facilitar a busca, crie conjuntos planos de todos os termos importantes
SET_TERMOS_PERSONAGENS = set()
for principal, dados in DIC_PERSONAGENS.items():
    SET_TERMOS_PERSONAGENS.add(principal.lower())
    for alias in dados["alias"]:
        SET_TERMOS_PERSONAGENS.add(alias.lower())

SET_TERMOS_LOCAIS = set()
for principal, dados in DIC_LOCAIS.items():
    SET_TERMOS_LOCAIS.add(principal.lower())
    for alias in dados["alias"]:
        SET_TERMOS_LOCAIS.add(alias.lower())

# Para eventos, você pode ter um set com as frases dos eventos ou as palavras-chave
SET_TERMOS_EVENTOS = set()
for evento_frase, palavras_chave in DIC_EVENTOS_PALAVRAS_CHAVE.items():
    SET_TERMOS_EVENTOS.add(evento_frase.lower())
    for palavra in palavras_chave:
        SET_TERMOS_EVENTOS.add(palavra.lower()) # Isso pode ser granular demais, adapte conforme a necessidade

In [55]:
def get_sentence_embedding(sentence_doc, model_w2v):
    """
    Calcula o embedding de uma sentença a partir dos embeddings das suas palavras.
    Ignora stop words e palavras que não estão no vocabulário do modelo.

    Args:
        sentence_doc (spacy.tokens.doc.Doc): Objeto Doc do spaCy para a sentença.
        model_w2v (gensim.models.Word2Vec): Modelo Word2Vec treinado.

    Returns:
        np.ndarray: O vetor de embedding da sentença.
    """
    tokens_in_vocab = [
        token.lemma_ for token in sentence_doc
        if token.is_alpha and not token.is_stop and token.lemma_ in model_w2v.wv
    ]
    if not tokens_in_vocab:
        return np.zeros(model_w2v.wv.vector_size)

    sentence_vec = np.mean([model_w2v.wv[token] for token in tokens_in_vocab], axis=0)
    return sentence_vec

# --- Função Principal de Geração de Chunks por Similaridade ---

def processar_texto_e_gerar_chunks_por_similaridade(
    texto,
    model_w2v,
    similaridade_limite=0.6,          # Limiar de similaridade para considerar uma quebra de chunk
    min_sentencas_chunk=3,            # Número mínimo de sentenças por chunk
    max_sentencas_chunk=15,           # Número máximo de sentenças por chunk (para evitar chunks gigantes)
    num_sentencas_sobreposicao=2      # Número de sentenças para sobreposição entre chunks
):
    """
    Processa um texto longo para gerar chunks com base na similaridade semântica entre sentenças,
    utilizando um modelo Word2Vec para embeddings.

    Args:
        texto (str): O texto longo a ser chunkado.
        model_w2v (gensim.models.Word2Vec): Modelo Word2Vec treinado para calcular embeddings de palavras.
        similaridade_limite (float): Limiar de similaridade de cosseno. Abaixo deste valor,
                                      uma quebra de chunk é considerada.
        min_sentencas_chunk (int): Número mínimo de sentenças que um chunk deve ter.
        max_sentencas_chunk (int): Número máximo de sentenças que um chunk pode ter.
        num_sentencas_sobreposicao (int): Número de sentenças do final de um chunk
                                          que serão incluídas no início do próximo.

    Returns:
        tuple: Uma tupla contendo:
            - chunks (list): Uma lista de strings, onde cada string é o texto original de um chunk.
            - mapa_chunks (dict): Um dicionário onde as chaves são os IDs dos chunks (int)
                                  e os valores são dicionários com 'texto_original',
                                  'tokens_processados' e 'entidades'.
    """
    chunks = []
    mapa_chunks = {}
    doc = nlp(texto) # Processa o texto completo com spaCy
    sentences_docs = list(doc.sents) # Obtém objetos de sentença do spaCy
    sentences_text = [sent.text for sent in sentences_docs] # Obtém o texto das sentenças

    if not sentences_docs: # Retorna vazio se não houver sentenças
        return [], {}

    # Pré-calcular embeddings para todas as sentenças (para otimização)
    sentence_embeddings = [get_sentence_embedding(sent_doc, model_w2v) for sent_doc in sentences_docs]

    current_chunk_sentences_text = []
    current_chunk_tokens_processed = []
    current_chunk_entities = {'PER': set(), 'LOC': set(), 'ORG': set(), 'MISC': set()}
    chunk_id_counter = 0

    def add_sentence_to_current_chunk(sent_doc_item, sent_text_item):
        """Adiciona uma sentença ao chunk atual, atualizando tokens e entidades."""
        nonlocal current_chunk_sentences_text, current_chunk_tokens_processed, current_chunk_entities
        current_chunk_sentences_text.append(sent_text_item)
        sent_tokens = [token.lemma_ for token in sent_doc_item if token.is_alpha and not token.is_stop]
        current_chunk_tokens_processed.extend(sent_tokens)
        for ent in sent_doc_item.ents:
            if ent.label_ in current_chunk_entities:
                current_chunk_entities[ent.label_].add(ent.text)

    def finalize_current_chunk():
        """Finaliza o chunk atual e o adiciona a mapa_chunks e chunks."""
        nonlocal chunk_id_counter, current_chunk_sentences_text, current_chunk_tokens_processed, current_chunk_entities
        if not current_chunk_sentences_text:
            return

        chunk_text_str = " ".join(current_chunk_sentences_text)
        chunk_tokens_str = " ".join(current_chunk_tokens_processed)

        mapa_chunks[chunk_id_counter] = {
            'texto_original': chunk_text_str,
            'tokens_processados': chunk_tokens_str.split(),
            'entidades': {k: list(v) for k, v in current_chunk_entities.items()}
        }
        chunks.append(chunk_text_str)
        chunk_id_counter += 1

    # Adiciona a primeira sentença para iniciar o primeiro chunk
    if sentences_docs: # Garante que há pelo menos uma sentença
        add_sentence_to_current_chunk(sentences_docs[0], sentences_text[0])

    # Itera a partir da segunda sentença
    for i in range(1, len(sentences_docs)):
        # Calcula o embedding do chunk atual (média dos embeddings das sentenças que já estão no chunk).
        # Pega os embeddings das sentenças que JÁ ESTÃO no current_chunk_sentences_text
        # A última sentença adicionada ao chunk foi no índice i-1 (se o loop está em i)
        # A primeira sentença no chunk seria no índice (i - len(current_chunk_sentences_text))
        start_idx_current_chunk = i - len(current_chunk_sentences_text)
        start_idx_current_chunk = max(0, start_idx_current_chunk) # Garante que o índice não seja negativo

        if not current_chunk_sentences_text:
            # Caso raro onde o chunk está vazio (apenas para a primeira iteração ou após um reset com 0 overlap)
            current_chunk_embedding = sentence_embeddings[i]
        else:
            current_chunk_embedding = np.mean(
                [sentence_embeddings[idx] for idx in range(start_idx_current_chunk, i)], # Todas as sentenças do chunk atual
                axis=0
            )

        # Calcula a similaridade entre o embedding do chunk atual e a próxima sentença
        similarity = cosine_similarity_manual(
            current_chunk_embedding,
            sentence_embeddings[i]
        )

        # Condição para finalizar um chunk:
        # 1. Similaridade cai abaixo do limiar E o chunk atingiu o tamanho mínimo OU
        # 2. O chunk atingiu o tamanho máximo permitido (para evitar chunks excessivamente longos)
        if (similarity < similaridade_limite and len(current_chunk_sentences_text) >= min_sentencas_chunk) or \
           (len(current_chunk_sentences_text) >= max_sentencas_chunk):

            finalize_current_chunk()

            # Reinicia o chunk atual e aplica a sobreposição
            current_chunk_sentences_text = []
            current_chunk_tokens_processed = []
            current_chunk_entities = {'PER': set(), 'LOC': set(), 'ORG': set(), 'MISC': set()}

            # Adiciona as sentenças de sobreposição ao novo chunk
            start_overlap_idx = max(0, i - num_sentencas_sobreposicao)
            for j in range(start_overlap_idx, i):
                add_sentence_to_current_chunk(sentences_docs[j], sentences_text[j])
            # A sentença atual (i) sempre é adicionada ao novo chunk
            add_sentence_to_current_chunk(sentences_docs[i], sentences_text[i])

            continue # Pula para a próxima iteração do loop

        # Se não for para finalizar o chunk, apenas adiciona a sentença atual
        add_sentence_to_current_chunk(sentences_docs[i], sentences_text[i])

    # Adiciona o último chunk se algo sobrou após o loop
    finalize_current_chunk()

    return chunks, mapa_chunks

In [56]:
# --- Função para Obter Vetor de Média de Palavras (para Word2Vec) ---
def obter_vetor_media(tokens, model, vector_size):
    vetores = [model.wv[token] for token in tokens if token in model.wv]
    if vetores:
        return np.mean(vetores, axis=0)
    else:
        return np.zeros(vector_size)

In [57]:
def preprocessar_pergunta_somente_lematizar(pergunta_str):
    pergunta_lower = pergunta_str.lower()
    doc = nlp(pergunta_lower)
    tokens_lematizados = [token.lemma_ for token in doc if token.is_alpha]
    return tokens_lematizados

# ## Função para Gerar Resposta Formatada (Atualizada com spaCy e Citação)

# Adicione pergunta_processada como um novo argumento
def gerar_resposta_formatada(pergunta, chunk_data, pergunta_processada): # <-- pergunta_processada já vem pré-processada

    texto_chunk = chunk_data['texto_original']
    pergunta_lower = pergunta.lower()

    # 1. Definir templates de resposta (AJUSTADO)
    templates = {
        "personagem": "Ao analisar a trajetória de {entidade}, o livro apresenta que {sumario}. Essa informação ajuda a compreender melhor aspectos como {coerencia}.",
        "evento": "Sobre o evento envolvendo {entidade}, a narrativa detalha que {sumario}. Isso evidencia {coerencia}.",
        "descricao": "O autor descreve {entidade} de forma detalhada. No trecho selecionado, a ênfase recai sobre {sumario}. Essa descrição ajuda a construir a imagem de {coerencia}.",
        "default": "Analisando o livro em busca de informações sobre \"{entidade}\", encontramos uma passagem relevante que afirma que {sumario}. Este trecho sugere que {coerencia}."
    }

    # 2. Extrair a entidade principal da pergunta (sem alteração)
    doc_pergunta = nlp(pergunta)
    # Prioriza entidades nomeadas se existirem, caso contrário, tenta noun_chunks
    entidade_principal = ""
    for ent in doc_pergunta.ents:
        if ent.label_ in ['PER', 'LOC', 'ORG', 'MISC']: # Considerar outras entidades nomeadas relevantes
            entidade_principal = ent.text
            break
    if not entidade_principal:
        noun_chunks = [chunk.text for chunk in doc_pergunta.noun_chunks]
        entidade_principal = noun_chunks[-1] if noun_chunks else pergunta


    # 3. Selecionar o template mais adequado (sem alteração)
    template_selecionado = templates["default"]
    if any(word in pergunta_lower for word in ['quem', 'o que fez', 'sobre', 'qual o papel']):
        template_selecionado = templates["personagem"]
    elif any(word in pergunta_lower for word in ['descreva', 'como era', 'costumes','como é descrita']):
        template_selecionado = templates["descricao"]
    elif any(word in pergunta_lower for word in ['aconteceu', 'evento', 'batalha', 'morte de']):
        template_selecionado = templates["evento"]


    # 4. Extrair citação e gerar sumário usando spaCy
    doc_chunk = nlp(texto_chunk)
    sentencas = [sent.text for sent in doc_chunk.sents]

    trecho_citacao = ""
    sumario_texto = ""

    # Define o limite de palavras para o trecho
    LIMITE_PALAVRAS = 150

    # Tenta encontrar a sentença mais relevante para a citação
    found_sentence_idx = -1
    # Modificado para usar 'pergunta_processada' (que é uma LISTA de lemas)
    pergunta_processada_str = " ".join(pergunta_processada) # Converte para string para usar 'in'
    for i, sent_text in enumerate(sentencas):
        # AQUI VOCÊ DEVE TER CUIDADO: `pergunta_processada` é uma lista de lemas.
        # Se você quer verificar se *algum lema* da pergunta está na sentença,
        # precisa iterar sobre os lemas da pergunta.
        # A linha original `any(word in sent_text.lower() for word in pergunta_processada)`
        # JÁ ESTÁ FAZENDO ISSO CORRETAMENTE SE `pergunta_processada` for uma lista de strings.
        # MANTEMOS COMO ESTÁ, POIS `pergunta_processada` será uma LISTA de lemas.
        if entidade_principal.lower() in sent_text.lower() or \
           any(word in sent_text.lower() for word in pergunta_processada):
            found_sentence_idx = i
            break

    # Se uma sentença relevante for encontrada, construa o trecho ao redor dela
    if found_sentence_idx != -1:
        # Começa com a sentença encontrada
        trecho_lista = [sentencas[found_sentence_idx]]

        # Expande para sentenças anteriores
        idx_anterior = found_sentence_idx - 1
        while idx_anterior >= 0 and len(" ".join(trecho_lista).split()) < LIMITE_PALAVRAS:
            trecho_lista.insert(0, sentencas[idx_anterior])
            idx_anterior -= 1

        # Expande para sentenças posteriores
        idx_posterior = found_sentence_idx + 1
        while idx_posterior < len(sentencas) and len(" ".join(trecho_lista).split()) < LIMITE_PALAVRAS:
            trecho_lista.append(sentencas[idx_posterior])
            idx_posterior += 1

        trecho_citacao = " ".join(trecho_lista)

        # Truncar o trecho se exceder o limite de palavras
        palavras_trecho = trecho_citacao.split()
        if len(palavras_trecho) > LIMITE_PALAVRAS:
            trecho_citacao = " ".join(palavras_trecho[:LIMITE_PALAVRAS]) + "..."

        # Para o sumário, podemos pegar as primeiras sentenças do trecho_citacao
        # Ou simplesmente a primeira sentença como antes, para ser mais conciso no corpo
        sumario_texto = sentencas[found_sentence_idx] if sentencas else "" # Pega a sentença chave para o sumário

    # Se não encontrar uma sentença relevante, usa o início do chunk até o limite de palavras
    elif sentencas:
        trecho_lista = []
        for sent_text in sentencas:
            trecho_lista.append(sent_text)
            if len(" ".join(trecho_lista).split()) >= LIMITE_PALAVRAS:
                break
        trecho_citacao = " ".join(trecho_lista)

        palavras_trecho = trecho_citacao.split()
        if len(palavras_trecho) > LIMITE_PALAVRAS:
            trecho_citacao = " ".join(palavras_trecho[:LIMITE_PALAVRAS]) + "..."

        sumario_texto = sentencas[0] if sentencas else "" # Sumário pega a primeira sentença do chunk

    # 5. Ajustar a coerência final (sem alteração)
    ajuste_coerencia = "a complexidade das relações e dos acontecimentos na obra."
    if template_selecionado == templates['personagem']:
        ajuste_coerencia = f"a personalidade e as motivações de {entidade_principal.split(' ')[-1]}."
    elif template_selecionado == templates['descricao']:
        ajuste_coerencia = "a atmosfera e o cenário em que a história se desenrola."

    # 6. Montar a resposta final (COM CITAÇÃO AO FINAL)
    if not trecho_citacao:
        return "Não foi possível gerar uma resposta formatada. O trecho encontrado é muito curto."

    resposta_formatada = template_selecionado.format(
        entidade=entidade_principal,
        sumario=f'o trecho \"{sumario_texto}\"', # Inclui o sumário no corpo da resposta
        coerencia=ajuste_coerencia
    )

    # Adiciona a citação explícita ao final
    resposta_formatada += f"\n\n**Trecho relevante do livro:** \"{trecho_citacao}\""

    return resposta_formatada

In [58]:
def responder_pergunta(pergunta, model_w2v, mapa_chunks, tfidf_vectorizer, tfidf_matrix, topn=3):
    pergunta_original_lower = pergunta.lower()

    pergunta_processada = preprocessar_pergunta_somente_lematizar(pergunta)

    doc_pergunta = nlp(pergunta_original_lower)

    vetor_pergunta_w2v = obter_vetor_media(pergunta_processada, model_w2v, model_w2v.vector_size)

    # === AQUI ESTÃO AS MUDANÇAS CRUCIAIS PARA O TF-IDF ===
    # Convertendo para array denso explicitamente antes de passar para a função manual
    vetor_pergunta_tfidf_dense = tfidf_vectorizer.transform([pergunta]).toarray()

    scores = []
    for idx, chunk_data in mapa_chunks.items():
        chunk_tokens = chunk_data['tokens_processados']
        chunk_texto_original = chunk_data['texto_original']
        chunk_texto_lower = chunk_texto_original.lower()
        vetor_chunk_w2v = obter_vetor_media(chunk_tokens, model_w2v, model_w2v.vector_size)

        # Para Word2Vec: já são arrays densos, então basta passar.
        if np.all(vetor_pergunta_w2v == 0) or np.all(vetor_chunk_w2v == 0):
            score_w2v = 0.0
        else:
            score_w2v = cosine_similarity_manual(vetor_pergunta_w2v, vetor_chunk_w2v)

        # === AQUI TAMBÉM É CRUCIAL PARA O TF-IDF ===
        # Convertendo chunk_tfidf_vector para array denso explicitamente
        chunk_tfidf_vector_dense = tfidf_matrix[idx].toarray()

        if np.all(vetor_pergunta_tfidf_dense == 0) or np.all(chunk_tfidf_vector_dense == 0):
            score_tfidf = 0.0
        else:
            # Passando os vetores TF-IDF JÁ DENSOS
            score_tfidf = cosine_similarity_manual(vetor_pergunta_tfidf_dense, chunk_tfidf_vector_dense)
        # ===============================================

        score_entidades_bonus = 0.0
        pergunta_entidades_relevantes = {'PER': set(), 'LOC': set(), 'EVENTO': set()}
        for ent in doc_pergunta.ents:
            if ent.label_ == 'PER': pergunta_entidades_relevantes['PER'].add(ent.text.lower())
            elif ent.label_ == 'LOC': pergunta_entidades_relevantes['LOC'].add(ent.text.lower())
        for termo_chave, dados in DIC_PERSONAGENS.items():
            if termo_chave.lower() in pergunta_original_lower: pergunta_entidades_relevantes['PER'].add(termo_chave.lower())
            for alias in dados["alias"]:
                if alias.lower() in pergunta_original_lower: pergunta_entidades_relevantes['PER'].add(alias.lower())
        for termo_chave, dados in DIC_LOCAIS.items():
            if termo_chave.lower() in pergunta_original_lower: pergunta_entidades_relevantes['LOC'].add(termo_chave.lower())
            for alias in dados["alias"]:
                if alias.lower() in pergunta_original_lower: pergunta_entidades_relevantes['LOC'].add(alias.lower())
        for evento_nome, palavras_chave in DIC_EVENTOS_PALAVRAS_CHAVE.items():
            if any(pc.lower() in pergunta_original_lower for pc in palavras_chave): pergunta_entidades_relevantes['EVENTO'].add(evento_nome.lower())
        entidades_chave_presentes_no_chunk = False
        for pessoa_pergunta in pergunta_entidades_relevantes['PER']:
            if pessoa_pergunta in chunk_texto_lower:
                score_entidades_bonus += 0.3
                entidades_chave_presentes_no_chunk = True
            elif any(alias.lower() in chunk_texto_lower for termo_principal, dados in DIC_PERSONAGENS.items() if pessoa_pergunta == termo_principal.lower() for alias in dados['alias']):
                score_entidades_bonus += 0.3
                entidades_chave_presentes_no_chunk = True
        for local_pergunta in pergunta_entidades_relevantes['LOC']:
            if local_pergunta in chunk_texto_lower:
                score_entidades_bonus += 0.2
                entidades_chave_presentes_no_chunk = True
            elif any(alias.lower() in chunk_texto_lower for termo_principal, dados in DIC_LOCAIS.items() if local_pergunta == termo_principal.lower() for alias in dados['alias']):
                score_entidades_bonus += 0.2
                entidades_chave_presentes_no_chunk = True
        for evento_pergunta in pergunta_entidades_relevantes['EVENTO']:
            palavras_chave_evento = DIC_EVENTOS_PALAVRAS_CHAVE.get(evento_pergunta.capitalize(), [])
            if any(pc.lower() in chunk_texto_lower for pc in palavras_chave_evento):
                score_entidades_bonus += 0.2
                entidades_chave_presentes_no_chunk = True
        score_combinado = (score_w2v * 0.5) + (score_tfidf * 0.3) + score_entidades_bonus
        pergunta_tem_entidade_importante = (len(pergunta_entidades_relevantes['PER']) > 0 or len(pergunta_entidades_relevantes['LOC']) > 0 or len(pergunta_entidades_relevantes['EVENTO']) > 0)
        if pergunta_tem_entidade_importante and not entidades_chave_presentes_no_chunk:
            score_combinado *= 0.4
        scores.append((idx, score_combinado))

    selecionados = sorted(scores, key=lambda x: x[1], reverse=True)

    if not selecionados or selecionados[0][1] < 0.3:
        print("\nDesculpe, não consegui encontrar uma resposta suficientemente relevante no livro.")
        return

    idx, score = selecionados[0]
    melhor_chunk = mapa_chunks[idx]

    resposta_final = gerar_resposta_formatada(pergunta, melhor_chunk, pergunta_processada)
    print(pergunta)
    print(pergunta_processada)

    print("\n" + "="*25)
    print("     RESPOSTA GERADA")
    print("="*25 + "\n")
    print(resposta_final)

In [59]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def setup_interactive_qa(responder_pergunta, model_w2v, mapa_chunks, tfidf_vectorizer, tfidf_matrix, topn=3):


    # Create the text input widget
    question_input = widgets.Textarea(
        value='',
        placeholder='Digite sua pergunta sobre o livro O Guarani aqui...',
        description='Pergunta:',
        disabled=False,
        layout=widgets.Layout(width='auto', height='80px')
    )

    # Create the output widget to display answers
    output_area = widgets.Output()

    # Create the button to submit the question
    submit_button = widgets.Button(
        description='Responder',
        disabled=False,
        button_style='success', # 'success', 'info', 'warning', 'danger' or ''
        tooltip='Clique para obter a resposta',
        icon='question' # (FontAwesome icons)
    )

    def on_button_click(b):
        with output_area:
            clear_output(wait=True) # Clear previous output
            pergunta_usuario = question_input.value
            if pergunta_usuario:
                print("Buscando resposta para:", pergunta_usuario)
                # Call your existing answer function
                responder_pergunta(pergunta_usuario, model_w2v, mapa_chunks, tfidf_vectorizer, tfidf_matrix, topn=topn)
            else:
                print("Por favor, digite uma pergunta.")
            question_input.value = '' # Clear the input field after submission

    submit_button.on_click(on_button_click)

    # Arrange the widgets
    ui = widgets.VBox([question_input, submit_button, output_area])
    display(ui)



In [60]:
setup_interactive_qa(responder_pergunta, model_w2v, mapa_chunks, tfidf_vectorizer, tfidf_matrix, topn=1)